### Day 7 Assignment: Delta Lake Internals & Ingestion Patterns 


### Basic Tasks 

#### 1. Create a table & use DESCRIBE HISTORY 

In [0]:
CREATE OR REPLACE TABLE dev.bronze.delta_sales_demo (
    order_id INT,
    customer_id INT,
    product_id INT,
    quantity INT,
    total_amount DOUBLE
)
USING DELTA;

In [0]:
INSERT INTO dev.bronze.delta_sales_demo VALUES
(1, 101, 1001, 2, 500.00),
(2, 102, 1002, 1, 250.00),
(3, 103, 1003, 3, 750.00);

In [0]:
-- change 1 : Insert into table
INSERT INTO dev.bronze.delta_sales_demo VALUES
(4, 104, 1004, 2, 400.00);

In [0]:
-- change 2 : Update the table 
UPDATE dev.bronze.delta_sales_demo
SET total_amount = 300.00
WHERE order_id = 2;

In [0]:
-- change 3 : Insert into table
INSERT INTO dev.bronze.delta_sales_demo VALUES
(5, 105, 1005, 1, 150.00);

In [0]:
DESCRIBE HISTORY dev.bronze.delta_sales_demo

#### 2. Incremental Loading with COPY INTO

In [0]:
CREATE TABLE IF NOT EXISTS dev.demo.sales_incremental;

COPY INTO dev.demo.sales_incremental
FROM "/Volumes/dev/demo/ex-volume/sales/"
FILEFORMAT = csv
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
select * from dev.demo.sales_incremental

In [0]:
DESCRIBE HISTORY dev.demo.sales_incremental

Created the Bronze table `dev.bronze.sales_incremental` and loaded two batches of CSV files incrementally using COPY INTO. The first batch loaded 78 rows and the second batch loaded 77 rows. Re-running the load for an already processed file confirmed that 'COPY INTO' does not reprocess the same file.

#### 3. Time Travel

In [0]:
SELECT *
FROM dev.demo.sales_incremental VERSION AS OF 1;

In [0]:
SELECT *
FROM dev.demo.sales_incremental TIMESTAMP AS OF '2026-08-27T05:09:35.000+00:00';

#### 4. Schema Evolution

Queried the previous state of `dev.demo.sales_incremental` using both `VERSION AS OF` and `TIMESTAMP AS OF`. Both queries returned the table state after the first batch was loaded, demonstrating Delta Lake's ability to access historical versions of data.

In [0]:
%python
new_data = spark.createDataFrame([
    (6, 106, 1006, 2, 400.0, 20.0)
], [
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "total_amount",
    "discount_amount"
])

In [0]:
%python
# Part 1 — Add a new column using mergeSchema
new_data.write\
    .format("delta")\
    .mode("append")\
    .option("mergeSchema", "true")\
    .saveAsTable("dev.bronze.delta_sales_demo")

In [0]:
DESCRIBE TABLE dev.bronze.delta_sales_demo; --discount_amount column is added

In [0]:
%python
# Part 2 — Change an existing column's type using overwriteSchema
df = spark.table("dev.bronze.delta_sales_demo")

df = df.withColumn(
    "discount_amount",
    df["discount_amount"].cast("decimal(10,2)")
)

In [0]:
%python
df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dev.bronze.delta_sales_demo")

In [0]:
DESCRIBE TABLE dev.bronze.delta_sales_demo; -- data type of discount_amount column has changed to decimal(10,2)

###  Intermediate Tasks

#### 5. Auto Loader

In [0]:
%python
# 1: Create the target Bronze table
source_path = "/Volumes/dev/demo/ex-volume/sales/"
target_table = "dev.bronze.sales_autoloader"
checkpoint_path = "/Volumes/dev/demo/ex-volume/checkpoints/sales_autoloader"
schema_path = "/Volumes/dev/demo/ex-volume/schema/sales_autoloader"

In [0]:
%python
# Step 2: Read the files using Auto Loader
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .load(source_path)
)

In [0]:
%python
# 3: Write to the Bronze Delta table
query = (
    df_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

In [0]:
SELECT COUNT(*) AS total_rows
FROM dev.bronze.sales_autoloader;

#### Auto Loader Incremental Ingestion

Configured Auto Loader to ingest CSV files from the sales folder into `dev.bronze.sales_autoloader`. After adding 'sales2' and 'sales3' files to the source folder, Auto Loader detected and ingested the new records. The row count increased from 78 to 155 to 233, confirming that the new files were processed incrementally.

![image_1787823344401.png](./image_1787823344401.png "image_1787823344401.png")

![image_1787823456911.png](./image_1787823456911.png "image_1787823456911.png")

![image_1787823502195.png](./image_1787823502195.png "image_1787823502195.png")

#### 6. RESTORE a Delta Table

In [0]:
DESCRIBE HISTORY dev.bronze.delta_sales_demo;

In [0]:
SELECT * FROM dev.bronze.delta_sales_demo VERSION AS OF 6;

In [0]:
RESTORE TABLE dev.bronze.delta_sales_demo
TO VERSION AS OF 5;

In [0]:
DESCRIBE TABLE dev.bronze.delta_sales_demo;

In [0]:
DESCRIBE HISTORY dev.bronze.delta_sales_demo;

Used DESCRIBE HISTORY to identify the last known-good version of the table and restored it using RESTORE TABLE. The restore creates a new Delta version with the data and schema from the selected historical version; it does not delete the versions created after that point. Those versions remain available for time travel until they are removed according to Delta's retention and VACUUM policies.

### Advanced Tasks 

#### 7. Comparison of Ingestion Patterns

**Batch CTAS** is simple and cost-effective for one-time or scheduled batch processing, but it requires the load to be executed again when new data arrives, making it less suitable for unpredictable file arrivals.

**COPY INTO** supports incremental file ingestion and tracks previously loaded files, preventing them from being processed again. However, the ingestion still needs to be triggered when new files need to be loaded.

**Auto Loader** is designed for incremental file ingestion at scale. It automatically detects new files and processes them without requiring the entire source directory to be reloaded, making it suitable for files that arrive unpredictably throughout the day.

**Lakeflow Declarative Pipelines** provides a managed framework for building complete data pipelines, including ingestion, transformations, dependencies, and monitoring. It is more suitable when file ingestion is part of a larger production pipeline rather than being used only for simple file ingestion.

**Recommendation:** For Cyntexa's unpredictable file source, **Auto Loader** is the preferred ingestion pattern. It provides low-latency incremental ingestion, avoids reprocessing previously loaded files, and reduces the need for manual intervention. If the requirement later expands into a complete production pipeline with multiple dependent transformations, Auto Loader can also be incorporated into a **Lakeflow Declarative Pipeline**.

#### 8. Recovery Runbook

In [0]:
DESCRIBE HISTORY dev.silver.sale_cln_tbl;

In [0]:
SELECT *
FROM dev.silver.sale_cln_tbl
VERSION AS OF 1; --check previous version

In [0]:
RESTORE TABLE dev.silver.sale_cln_tbl
TO VERSION AS OF 1; -- restore to that version if the previous version seems correct

In [0]:
SELECT COUNT(*)
FROM dev.silver.sale_cln_tbl; --validate 

If a bad file corrupts the Silver table, the on-call engineer first runs `DESCRIBE HISTORY` to identify when the bad data was introduced and determine the last known-good version. The previous version is validated using Delta time travel with `VERSION AS OF`.

If the entire table needs to be rolled back, `RESTORE TABLE ... TO VERSION AS OF` is used. The restored table is then validated using row counts and key business metrics before the pipeline is resumed. RESTORE creates a new Delta version while preserving the previous versions in the transaction history.

If a full restore is not appropriate, the last known-good version can be read using time travel and written back to the table using an overwrite operation.

#### 9. Data Freshness Report

In [0]:
DESCRIBE HISTORY dev.demo.sales_incremental;


Used DESCRIBE HISTORY on the Delta table `dev.bronze.sales_incremental` to review when the table was actually updated.

The table history shows the following data-ingestion activity:

- **Version 1:** COPY INTO operation at 05:09:35 UTC, loading 1 file and 78 rows.
- **Version 2:** COPY INTO operation at 05:17:04 UTC, loading 1 file and 77 rows.

The two ingestion operations occurred approximately **7.5 minutes apart**, showing that the table was refreshed incrementally during the observed period.

The history also shows that each COPY INTO operation processed one file, confirming that new files were being added rather than the existing data being fully reloaded.

#### SLA Validation

'DESCRIBE HISTORY' provides a reliable record of when changes were committed to the Delta table. These timestamps can be compared with the freshness SLA agreed with the business stakeholder.

For example, if the expected SLA is that the table should be updated at least every **15 minutes**, the observed 7.5-minute interval satisfies that requirement for this period. However, this observation alone is not enough to prove that the SLA is consistently met; a longer period of history should be reviewed to establish the actual update frequency.

Therefore, the table history can be used as an audit trail for validating data freshness and identifying periods where expected updates may have been delayed or missing.